# LeetCode 495 — Teemo Attacking bằng Python

**Link đề:** https://leetcode.com/problems/teemo-attacking/
**Độ khó:** Easy · **Chủ đề:** Array, Simulation (mô phỏng), hợp các khoảng (interval union)

```python
class Solution:
    def findPoisonedDuration(self, timeSeries: list[int], duration: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đề: mỗi đòn đánh tại giây `t` gây độc trong **đoạn đóng** `[t, t + duration − 1]`; đòn mới **reset** thời gian độc. Tính tổng số giây bị độc.
2. Ôn `set`, `range`, `zip(a, a[1:])` để duyệt **cặp liền kề**, `min`, `sum` với generator.
3. Đi từ mô phỏng từng giây bằng `set` (chậm khi `duration` lớn) → **gộp khoảng** giữ `end` → công thức gọn `Σ min(khoảng cách, duration) + duration`.
4. Trace, chứng minh tính đúng, phân tích Time/Space; xử lý biên `duration = 0`, đòn trùng thời điểm.
5. Nhận dạng pattern "hợp các khoảng đã sắp xếp", làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Teemo tấn công Ashe bằng đòn độc. Đòn tại giây `t` làm Ashe bị độc **đúng `duration` giây**: các giây `t, t+1, ..., t + duration − 1`. Nếu Teemo đánh lại **trước khi** độc hết, đồng hồ độc được **đặt lại**: độc sẽ hết sau `duration` giây tính từ đòn mới.

Cho mảng **không giảm** `timeSeries` (thời điểm các đòn) và số `duration`. Trả về **tổng số giây** Ashe bị độc.

| `timeSeries` | `duration` | Giây bị độc | Output |
|---|:---:|---|:---:|
| `[1,4]` | 2 | 1, 2, 4, 5 | 4 |
| `[1,2]` | 2 | 1, 2, 3 | 3 |

**Constraints:** `1 <= timeSeries.length <= 10^4`, `0 <= timeSeries[i], duration <= 10^7`, `timeSeries` sắp xếp không giảm.

Chú ý: `duration` có thể bằng **0** (đòn không gây độc), và hai đòn có thể ở **cùng một giây**.

In [1]:
t, duration = 1, 2
print("Đòn tại giây", t, "-> độc các giây:", list(range(t, t + duration)))
t, duration = 4, 2
print("Đòn tại giây", t, "-> độc các giây:", list(range(t, t + duration)))

Đòn tại giây 1 -> độc các giây: [1, 2]
Đòn tại giây 4 -> độc các giây: [4, 5]


`range(t, t + duration)` sinh đúng `duration` số: `t` đến `t + duration − 1` — khớp với đoạn đóng `[t, t + duration − 1]` của đề.

Câu hỏi đặt ra trước khi code:
- Hai đòn **cách nhau xa** (khoảng cách ≥ `duration`) thì tổng độc là bao nhiêu?
- Hai đòn **gần nhau** thì phần nào bị đếm trùng?
- Có cần liệt kê từng giây không, khi `duration` có thể tới `10^7`?

## 2. Nền tảng Python cần biết

### 2.1. `set` — tập hợp tự loại trùng

`set.update(iterable)` thêm mọi phần tử; phần tử đã có thì bỏ qua. `len(s)` cho số phần tử **khác nhau**.

In [2]:
poisoned = set()
poisoned.update(range(1, 3))     # đòn tại 1, duration 2 -> {1, 2}
print(poisoned)
poisoned.update(range(2, 4))     # đòn tại 2 -> {2, 3}, số 2 đã có
print(poisoned, "-> số giây:", len(poisoned))

{1, 2}
{1, 2, 3} -> số giây: 3


### 2.2. Duyệt các cặp liền kề bằng `zip(a, a[1:])`

`a[1:]` là list bỏ phần tử đầu. `zip(a, a[1:])` ghép `a[0]` với `a[1]`, `a[1]` với `a[2]`, … → mỗi cặp **(đòn trước, đòn sau)**. `zip` dừng ở list ngắn hơn nên không vượt chỉ số. (Python 3.10+ còn có `itertools.pairwise(a)` làm đúng việc này.)

In [3]:
a = [1, 4, 5, 10]
print(list(zip(a, a[1:])))
print("khoảng cách:", [nxt - cur for cur, nxt in zip(a, a[1:])])
from itertools import pairwise
print("pairwise:", list(pairwise(a)))

[(1, 4), (4, 5), (5, 10)]
khoảng cách: [3, 1, 5]
pairwise: [(1, 4), (4, 5), (5, 10)]


### 2.3. `min` và `sum` với generator

`sum(min(g, d) for g in gaps)` cộng từng `min(g, d)` mà không tạo list trung gian.

In [4]:
gaps = [3, 1, 5]
d = 2
print([min(g, d) for g in gaps], "-> tổng", sum(min(g, d) for g in gaps))

[2, 1, 2] -> tổng 5


## 3. Cách 1 — Mô phỏng từng giây bằng `set`

Làm đúng như định nghĩa: mỗi đòn thêm các giây `t .. t + duration − 1` vào tập; đáp án là kích thước tập.

Đúng, nhưng tốn `O(n · duration)` thời gian và bộ nhớ. Với `n = 10^4`, `duration = 10^7` → `10^11` phần tử: **không thể** chạy. Chỉ dùng làm oracle với input nhỏ.

In [5]:
def poisoned_set(timeSeries: list[int], duration: int) -> int:
    seconds = set()
    for t in timeSeries:
        seconds.update(range(t, t + duration))
    return len(seconds)

print(poisoned_set([1, 4], 2))   # 4
print(poisoned_set([1, 2], 2))   # 3
print(poisoned_set([1, 2], 0))   # 0

4
3
0


### 3.1. Thấy nút thắt bằng số liệu

In [6]:
import time
for duration in [10**3, 10**4, 10**5]:
    series = list(range(0, 100 * duration, duration // 2))[:100]   # 100 đòn chồng lấn một nửa
    t0 = time.perf_counter()
    r = poisoned_set(series, duration)
    print(f"duration={duration:>7}: {r:>9} giây độc, mô phỏng mất {time.perf_counter() - t0:.4f}s")

duration=   1000:     50500 giây độc, mô phỏng mất 0.0067s
duration=  10000:    505000 giây độc, mô phỏng mất 0.0530s


duration= 100000:   5050000 giây độc, mô phỏng mất 0.6132s


Thời gian tăng tỉ lệ với `duration` (và bộ nhớ cũng vậy). Ta không cần biết **từng giây** — chỉ cần **độ dài** các đoạn bị độc.

## 4. Insight — mỗi đòn đóng góp `min(khoảng cách tới đòn sau, duration)`

Xét đòn tại `t_i` và đòn kế tiếp `t_{i+1}`, khoảng cách `gap = t_{i+1} − t_i`:
- Nếu `gap >= duration`: độc của đòn `i` hết **trước** đòn sau → đóng góp đủ `duration` giây.
- Nếu `gap < duration`: đòn sau reset đồng hồ → đòn `i` chỉ "sở hữu" `gap` giây (từ `t_i` đến `t_{i+1} − 1`); phần còn lại được tính cho đòn sau.

Gộp lại: đòn `i` đóng góp `min(gap, duration)`. **Đòn cuối cùng** không bị ai cắt → đóng góp đủ `duration`.

> `answer = Σ_{i=0}^{n-2} min(t_{i+1} − t_i, duration) + duration`

Kiểm tra ví dụ 2: `[1, 2]`, `duration = 2`: `min(1, 2) + 2 = 3` ✔. Ví dụ 1: `[1, 4]`: `min(3, 2) + 2 = 4` ✔.

In [7]:
ts, d = [1, 2], 2
gap = ts[1] - ts[0]
print("gap =", gap, "| đòn 1 đóng góp min(gap, d) =", min(gap, d), "| đòn cuối đóng góp", d, "| tổng =", min(gap, d) + d)

gap = 1 | đòn 1 đóng góp min(gap, d) = 1 | đòn cuối đóng góp 2 | tổng = 3


## 5. Lời giải chính — mổ xẻ từng đoạn rồi ghép

### 5.1. Trường hợp `duration == 0`
Công thức tự cho `0` (mọi `min(gap, 0) = 0`, cộng `0`) — không cần xử lý riêng, nhưng nên **test**.

### 5.2. Cộng phần "bị cắt" của các đòn (trừ đòn cuối)
```python
total = 0
for i in range(len(timeSeries) - 1):
    total += min(timeSeries[i + 1] - timeSeries[i], duration)
```
`range(len - 1)` để `i + 1` luôn hợp lệ.

### 5.3. Cộng đòn cuối
```python
return total + duration
```
`timeSeries` có ít nhất 1 phần tử nên luôn có "đòn cuối".

In [8]:
def poisoned_gaps(timeSeries: list[int], duration: int) -> int:
    total = 0
    for i in range(len(timeSeries) - 1):
        total += min(timeSeries[i + 1] - timeSeries[i], duration)
    return total + duration

print(poisoned_gaps([1, 4], 2), poisoned_gaps([1, 2], 2), poisoned_gaps([1, 2], 0), poisoned_gaps([5], 3))

4 3 0 3


## 6. TRACE từng bước

Ví dụ đủ "khó": `timeSeries = [1, 2, 3, 7, 7, 20]`, `duration = 4` (có đòn sát nhau, đòn trùng giây, đòn cách xa).

| i | `t_i` | `t_{i+1}` | gap | `min(gap, 4)` | ý nghĩa | total |
|---|---|---|---|---|---|---|
| 0 | 1 | 2 | 1 | 1 | bị đòn sau reset | 1 |
| 1 | 2 | 3 | 1 | 1 | bị reset | 2 |
| 2 | 3 | 7 | 4 | 4 | vừa hết độc đúng lúc | 6 |
| 3 | 7 | 7 | 0 | 0 | đòn trùng giây, không thêm gì | 6 |
| 4 | 7 | 20 | 13 | 4 | hết độc từ lâu | 10 |
| cuối | 20 | — | — | +4 | đòn cuối đủ `duration` | **14** |

Kiểm tra bằng tay các giây độc: `1..6` (6 giây), `7..10` (4 giây), `20..23` (4 giây) → 14 ✔.

In [9]:
def trace_poisoned(timeSeries, duration):
    total = 0
    for i in range(len(timeSeries) - 1):
        cur, nxt = timeSeries[i], timeSeries[i + 1]
        gap = nxt - cur
        add = min(gap, duration)
        reason = "đủ duration (hết độc trước đòn sau)" if gap >= duration else f"bị reset sau {gap} giây"
        total += add
        print(f"i={i}: đòn {cur} -> đòn {nxt}, gap={gap:>2}, cộng {add} ({reason}), total={total}")
    total += duration
    print(f"đòn cuối tại {timeSeries[-1]}: cộng {duration} -> total = {total}")
    return total

trace_poisoned([1, 2, 3, 7, 7, 20], 4)
print("Đối chiếu mô phỏng set:", poisoned_set([1, 2, 3, 7, 7, 20], 4))

i=0: đòn 1 -> đòn 2, gap= 1, cộng 1 (bị reset sau 1 giây), total=1
i=1: đòn 2 -> đòn 3, gap= 1, cộng 1 (bị reset sau 1 giây), total=2
i=2: đòn 3 -> đòn 7, gap= 4, cộng 4 (đủ duration (hết độc trước đòn sau)), total=6
i=3: đòn 7 -> đòn 7, gap= 0, cộng 0 (bị reset sau 0 giây), total=6
i=4: đòn 7 -> đòn 20, gap=13, cộng 4 (đủ duration (hết độc trước đòn sau)), total=10
đòn cuối tại 20: cộng 4 -> total = 14
Đối chiếu mô phỏng set: 14


## 7. Cách 3 — Gộp khoảng: giữ `end` là giây cuối cùng còn độc

Góc nhìn "hợp các đoạn": mỗi đòn tạo đoạn `[t, t + duration)` (nửa mở). Duyệt theo thời gian, giữ `end` = thời điểm **hết độc** hiện tại (độc trong `[.., end)`).
- Nếu `t >= end`: đoạn mới tách rời → cộng `duration`.
- Nếu `t < end`: chồng lấn → chỉ cộng phần **kéo dài thêm** `t + duration − end`.
- Cập nhật `end = t + duration`.

Cách này tổng quát hơn (dùng được khi mỗi đòn có `duration` khác nhau — xem biến thể).

In [10]:
def poisoned_merge(timeSeries: list[int], duration: int) -> int:
    total = 0
    end = 0                      # độc hiện tại kéo dài tới trước giây 'end'
    for t in timeSeries:
        new_end = t + duration
        if t >= end:
            total += duration            # tách rời
        else:
            total += new_end - end       # chỉ phần kéo dài thêm
        end = new_end
    return total

print(poisoned_merge([1, 4], 2), poisoned_merge([1, 2], 2), poisoned_merge([1, 2, 3, 7, 7, 20], 4))

4 3 14


In [11]:
def trace_merge(timeSeries, duration):
    total, end = 0, 0
    for t in timeSeries:
        new_end = t + duration
        if t >= end:
            add = duration; kind = "tách rời"
        else:
            add = new_end - end; kind = f"chồng lấn, kéo dài từ {end} tới {new_end}"
        total += add
        print(f"đòn t={t:>2}: end cũ={end:>2}, {kind:<32} +{add} -> total={total}, end mới={new_end}")
        end = new_end
    return total

trace_merge([1, 2, 3, 7, 7, 20], 4)

đòn t= 1: end cũ= 0, tách rời                         +4 -> total=4, end mới=5
đòn t= 2: end cũ= 5, chồng lấn, kéo dài từ 5 tới 6    +1 -> total=5, end mới=6
đòn t= 3: end cũ= 6, chồng lấn, kéo dài từ 6 tới 7    +1 -> total=6, end mới=7
đòn t= 7: end cũ= 7, tách rời                         +4 -> total=10, end mới=11
đòn t= 7: end cũ=11, chồng lấn, kéo dài từ 11 tới 11  +0 -> total=10, end mới=11
đòn t=20: end cũ=11, tách rời                         +4 -> total=14, end mới=24


14

Chú ý: vì `timeSeries` không giảm và mọi đòn có cùng `duration`, `new_end >= end` luôn đúng → `end = new_end` không bao giờ làm lùi `end`. Nếu `duration` mỗi đòn khác nhau thì phải dùng `end = max(end, new_end)`.

## 8. Cách 4 — Một dòng với `zip`

In [12]:
def poisoned_oneliner(timeSeries: list[int], duration: int) -> int:
    return sum(min(b2 - a, duration) for a, b2 in zip(timeSeries, timeSeries[1:])) + duration

print(poisoned_oneliner([1, 4], 2), poisoned_oneliner([1, 2], 2), poisoned_oneliner([1, 2, 3, 7, 7, 20], 4))

4 3 14


`timeSeries[1:]` tạo bản sao `O(n)` bộ nhớ; nếu muốn `O(1)` dùng `itertools.pairwise` hoặc vòng `for` chỉ số như mục 5.

## 9. Code nộp LeetCode (Python 3)

In [13]:
class Solution:
    def findPoisonedDuration(self, timeSeries: list[int], duration: int) -> int:
        total = 0
        for i in range(len(timeSeries) - 1):
            total += min(timeSeries[i + 1] - timeSeries[i], duration)
        return total + duration

sol = Solution()
print(sol.findPoisonedDuration([1, 4], 2))   # 4
print(sol.findPoisonedDuration([1, 2], 2))   # 3

4
3


## 10. Vì sao đúng?

Đặt `P` = tập các giây bị độc. Với đòn `i` (không phải đòn cuối), định nghĩa "phần sở hữu" `S_i = [t_i, min(t_i + duration, t_{i+1}))` — các giây độc kể từ đòn `i` **trước khi** đòn `i+1` xảy ra. Đòn cuối sở hữu `S_{n-1} = [t_{n-1}, t_{n-1} + duration)`.

- **Không chồng nhau:** `S_i` nằm trong `[t_i, t_{i+1})`, các nửa khoảng này rời nhau vì `t` không giảm.
- **Phủ đủ `P`:** một giây `x` bị độc ⇔ đòn gần nhất `t_i <= x` (đòn cuối cùng xảy ra không muộn hơn `x`) thỏa `x < t_i + duration` (do cơ chế reset, chỉ đòn **gần nhất** quyết định). Khi đó `x ∈ [t_i, t_{i+1})` và `x < t_i + duration` → `x ∈ S_i`. Ngược lại mọi phần tử của `S_i` đều độc.
- `|S_i| = min(duration, t_{i+1} − t_i)` và `|S_{n-1}| = duration`.

Vậy `|P| = Σ |S_i|` đúng bằng công thức. Đòn trùng giây (`gap = 0`) có `S_i` rỗng — đúng là không đóng góp gì thêm.

## 11. Phân tích Time / Space

Gọi `n = len(timeSeries)`, `D = duration`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng bằng `set` | `O(n · D)` | `O(n · D)` | không chạy nổi khi `D = 10^7` |
| 2. `Σ min(gap, D) + D` | `O(n)` | `O(1)` | **chuẩn** |
| 3. Gộp khoảng với `end` | `O(n)` | `O(1)` | tổng quát hơn |
| 4. Một dòng `zip` | `O(n)` | `O(n)` | do `timeSeries[1:]` |

Về độ lớn đáp án: các giây độc đều nằm trong `[0, t_max + D)`, nên đáp án `<= t_max + D <= 2·10^7` — vừa kiểu `int` 32 bit ở C/Java; trong Python số nguyên không bao giờ tràn.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", [1, 4], 2, 4),
    ("đề 2", [1, 2], 2, 3),
    ("một đòn", [5], 3, 3),
    ("duration = 0", [1, 2, 3], 0, 0),
    ("đòn trùng giây", [1, 1, 1], 5, 5),
    ("bắt đầu từ giây 0", [0, 1], 1, 2),
    ("cách đúng duration", [1, 3, 5], 2, 6),
    ("ví dụ trace", [1, 2, 3, 7, 7, 20], 4, 14),
    ("duration rất lớn", [0, 10**7], 10**7, 2 * 10**7),
]
sol = Solution()
for name, ts, d, expected in cases:
    check(name, sol.findPoisonedDuration(ts, d), expected)

PASS  đề 1: 4
PASS  đề 2: 3
PASS  một đòn: 3
PASS  duration = 0: 0
PASS  đòn trùng giây: 5
PASS  bắt đầu từ giây 0: 2
PASS  cách đúng duration: 6
PASS  ví dụ trace: 14
PASS  duration rất lớn: 20000000


In [15]:
import random
random.seed(495)
methods = [poisoned_set, poisoned_gaps, poisoned_merge, poisoned_oneliner]
for _ in range(2000):
    n = random.randint(1, 8)
    ts = sorted(random.randint(0, 20) for _ in range(n))
    d = random.randint(0, 6)
    results = {f(ts, d) for f in methods}
    assert len(results) == 1, (ts, d, results)
print("2000 bộ ngẫu nhiên (có đòn trùng, duration = 0): 4 cách khớp với mô phỏng")

big = list(range(0, 2 * 10**4, 2))     # 10^4 đòn
print("10^4 đòn, duration 10^7:", Solution().findPoisonedDuration(big, 10**7))

2000 bộ ngẫu nhiên (có đòn trùng, duration = 0): 4 cách khớp với mô phỏng
10^4 đòn, duration 10^7: 10019998


Với `big`: các đòn cách nhau 2 giây, `duration` khổng lồ → mỗi đòn (trừ cuối) góp 2 giây, đòn cuối góp `10^7`: `2 · 9999 + 10^7 = 10019998`. Mô phỏng bằng `set` ở đây sẽ cần hàng tỉ phần tử.

## 13. Lỗi người mới hay mắc

### Lỗi 1: cộng `duration` cho mọi đòn
Quên phần chồng lấn → đếm trùng.

In [16]:
def wrong_sum_all(ts, d):
    return len(ts) * d

print("sai :", wrong_sum_all([1, 2], 2))
print("đúng:", poisoned_gaps([1, 2], 2))

sai : 4
đúng: 3


### Lỗi 2: lệch 1 khi hiểu đoạn độc là `[t, t + duration]` (đóng cả hai đầu)
Đề nói `[t, t + duration − 1]`. Dùng `t + duration` như giây cuối sẽ làm mỗi đoạn dài `duration + 1`.

In [17]:
def wrong_inclusive(ts, d):
    s = set()
    for t in ts:
        s.update(range(t, t + d + 1))     # SAI: thêm 1 giây
    return len(s)

print("sai :", wrong_inclusive([1, 4], 2))
print("đúng:", poisoned_set([1, 4], 2))

sai : 6
đúng: 4


### Lỗi 3: quên cộng đòn cuối
Vòng `for i in range(len - 1)` chỉ xét các đòn **có đòn sau**; thiếu `+ duration` sẽ ra `0` cho mảng một phần tử.

In [18]:
def wrong_no_last(ts, d):
    return sum(min(ts[i + 1] - ts[i], d) for i in range(len(ts) - 1))

print("sai :", wrong_no_last([5], 3), "| đúng:", poisoned_gaps([5], 3))

sai : 0 | đúng: 3


### Lỗi 4: trộn hai quy ước "đoạn đóng" và "nửa mở"
- Quy ước **nửa mở** (mục 7): `end = t + duration` là thời điểm **hết** độc; đòn tách rời khi `t >= end`.
- Quy ước **đóng** (bài tập 3): `end = t + duration − 1` là **giây cuối** còn độc; đòn tách rời khi `t > end`.

Lấy `end` theo quy ước này nhưng điều kiện theo quy ước kia sẽ lệch 1 giây ở mỗi chỗ hai đoạn "chạm nhau". Hãy chọn một quy ước, ghi chú ngay cạnh biến `end`, và giữ nhất quán.

### Lỗi 5: dùng `range(len(ts))` và đọc `ts[i + 1]`
`IndexError` ở phần tử cuối.

In [19]:
try:
    ts = [1, 4]
    total = 0
    for i in range(len(ts)):
        total += min(ts[i + 1] - ts[i], 2)
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 14. Mẹo nhận dạng pattern

- "Nhiều khoảng thời gian **đã sắp xếp**, hỏi tổng độ dài phần hợp" → duyệt một lượt, giữ `end` hiện tại (sweep line).
- Mọi khoảng **cùng độ dài** và điểm bắt đầu tăng dần → công thức `Σ min(gap, len) + len`.
- Cơ chế "reset đồng hồ" = phần sở hữu của mỗi sự kiện bị cắt bởi sự kiện kế tiếp.

Bài cùng họ: 56 Merge Intervals, 57 Insert Interval, 252/253 Meeting Rooms (Premium), 435 Non-overlapping Intervals, 1854 Maximum Population Year, 2406 Divide Intervals Into Minimum Number of Groups.

## 15. Biến thể

### 15.1. Mỗi đòn có `duration` riêng (chưa chắc tăng dần)
Gộp khoảng với `end = max(end, new_end)`.

In [20]:
def poisoned_varying(attacks: list[tuple[int, int]]) -> int:
    # attacks: list (t, d), đã sắp theo t
    total, end = 0, 0
    for t, d in attacks:
        new_end = t + d
        if t >= end:
            total += d
            end = new_end
        elif new_end > end:
            total += new_end - end
            end = new_end
        # nếu new_end <= end: đòn này nằm gọn trong độc cũ, không thêm gì
    return total

print(poisoned_varying([(1, 5), (2, 1), (4, 4)]))   # [1,6) và [4,8) -> 1..7 = 7 giây

7


### 15.2. Trả về các **đoạn** bị độc thay vì tổng

In [21]:
def poisoned_intervals(ts, d):
    intervals = []
    for t in ts:
        if d == 0:
            continue
        if intervals and t <= intervals[-1][1]:       # chồng/chạm đoạn trước (đoạn đóng)
            intervals[-1][1] = max(intervals[-1][1], t + d - 1)
        else:
            intervals.append([t, t + d - 1])
    return intervals

print(poisoned_intervals([1, 2, 3, 7, 7, 20], 4))

[[1, 6], [7, 10], [20, 23]]


`intervals[-1]` là đoạn cuối cùng (list `[đầu, cuối]`), sửa trực tiếp `intervals[-1][1]` để kéo dài nó. Đoạn `[7, 10]` nối liền `[1, 6]` nhưng không chồng (7 > 6), nên được giữ riêng; tổng độ dài `6 + 4 + 4 = 14` vẫn khớp.

## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Tính tay
`timeSeries = [2, 3, 8, 9, 10]`, `duration = 3`. Lập bảng gap, `min(gap, 3)`, tổng.

### Bài 2 — Hoàn thiện công thức

In [22]:
def ex2_poisoned(timeSeries: list[int], duration: int) -> int:
    # TODO 1: total = 0
    # TODO 2: với mỗi cặp liền kề cộng min(gap, duration)
    # TODO 3: cộng duration cho đòn cuối
    pass

# assert ex2_poisoned([1, 4], 2) == 4
# assert ex2_poisoned([1, 2], 2) == 3

### Bài 3 — Viết lại bằng cách gộp khoảng (giữ `end`) với quy ước **đoạn đóng** `end = t + duration − 1`
Cẩn thận ca `duration = 0`.

In [23]:
def ex3_poisoned_closed(timeSeries: list[int], duration: int) -> int:
    # TODO
    pass

# assert ex3_poisoned_closed([1, 4], 2) == 4
# assert ex3_poisoned_closed([1, 2], 2) == 3
# assert ex3_poisoned_closed([1, 2], 0) == 0

### Bài 4 — Merge Intervals (LeetCode 56)
Cho list đoạn `[start, end]` (đóng, chưa sắp xếp), gộp các đoạn chồng nhau. Ví dụ `[[1,3],[2,6],[8,10],[15,18]]` → `[[1,6],[8,10],[15,18]]`; `[[1,4],[4,5]]` → `[[1,5]]`.

In [24]:
def ex4_merge(intervals: list[list[int]]) -> list[list[int]]:
    # TODO: sắp xếp theo start rồi gộp
    pass

# assert ex4_merge([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
# assert ex4_merge([[1, 4], [4, 5]]) == [[1, 5]]

### Bài 5 — Mini challenge: `duration` nhỏ nhất để độc ít nhất `need` giây
Cho `timeSeries` và `need`, tìm `duration` nhỏ nhất sao cho tổng độc `>= need`. Tổng độc **tăng** theo `duration` → tìm kiếm nhị phân. Ví dụ `[1, 4], need = 4` → `2`; `[1, 2, 3], need = 5` → `3`.

In [25]:
def ex5_min_duration(timeSeries: list[int], need: int) -> int:
    # TODO: binary search duration trong [0, need]
    pass

# assert ex5_min_duration([1, 4], 4) == 2
# assert ex5_min_duration([1, 2, 3], 5) == 3
# assert ex5_min_duration([7], 0) == 0

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: gaps = `1, 5, 1, 1`. Mức 2: `min` với 3 → `1, 3, 1, 1`. Mức 3: tổng 6 + đòn cuối 3.

**Bài 2** — Mức 1: `range(len(ts) - 1)`. Mức 2: `min(ts[i+1] - ts[i], duration)`. Mức 3: `return total + duration`.

**Bài 3** — Mức 1: nếu `duration == 0` trả 0 ngay. Mức 2: khởi tạo `end = -1` (chưa có độc). Mức 3: `if t > end: total += duration` else `total += t + duration - 1 - end`; rồi `end = t + duration - 1`.

**Bài 4** — Mức 1: `intervals.sort()` (sắp theo phần tử đầu). Mức 2: so `start <= merged[-1][1]`. Mức 3: chồng thì `merged[-1][1] = max(merged[-1][1], end)`, không thì `append`.

**Bài 5** — Mức 1: `f(d)` = tổng độc là hàm không giảm. Mức 2: `lo, hi = 0, need` (vì `d = need` luôn đủ: đòn cuối đã độc `need` giây). Mức 3: `while lo < hi: mid = (lo + hi) // 2; if f(mid) >= need: hi = mid else: lo = mid + 1`.

## 17. Đáp án tham khảo

**Bài 1:** gaps `1, 5, 1, 1` → `min(·, 3)` = `1, 3, 1, 1` → tổng 6, cộng đòn cuối 3 → **9**. Kiểm tra bằng giây: đòn 2 và 3 cho độc `2..5` (4 giây), các đòn 8, 9, 10 cho độc `8..12` (5 giây) → 9 ✔.

In [26]:
def ans2_poisoned(ts, d):
    total = 0
    for i in range(len(ts) - 1):
        total += min(ts[i + 1] - ts[i], d)
    return total + d

def ans3_poisoned_closed(ts, d):
    if d == 0:
        return 0
    total, end = 0, -1
    for t in ts:
        last = t + d - 1
        if t > end:
            total += d
        else:
            total += last - end
        end = last
    return total

def ans4_merge(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

def ans5_min_duration(ts, need):
    lo, hi = 0, need
    while lo < hi:
        mid = (lo + hi) // 2
        if ans2_poisoned(ts, mid) >= need:
            hi = mid
        else:
            lo = mid + 1
    return lo

print("bài 1:", ans2_poisoned([2, 3, 8, 9, 10], 3), "| mô phỏng set:", poisoned_set([2, 3, 8, 9, 10], 3))
assert ans2_poisoned([2, 3, 8, 9, 10], 3) == poisoned_set([2, 3, 8, 9, 10], 3)
assert ans2_poisoned([1, 4], 2) == 4 and ans2_poisoned([1, 2], 2) == 3
assert ans3_poisoned_closed([1, 4], 2) == 4 and ans3_poisoned_closed([1, 2], 2) == 3 and ans3_poisoned_closed([1, 2], 0) == 0
for _ in range(500):
    ts = sorted(random.randint(0, 15) for _ in range(random.randint(1, 6)))
    d = random.randint(0, 5)
    assert ans3_poisoned_closed(ts, d) == poisoned_set(ts, d)
assert ans4_merge([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert ans4_merge([[1, 4], [4, 5]]) == [[1, 5]]
assert ans5_min_duration([1, 4], 4) == 2
assert ans5_min_duration([1, 2, 3], 5) == 3
assert ans5_min_duration([7], 0) == 0
print("Đáp án bài 1–5: PASS")

bài 1: 9 | mô phỏng set: 9
Đáp án bài 1–5: PASS


## 18. Template nhớ nhanh + checklist submit

```python
total = 0
for i in range(len(timeSeries) - 1):
    total += min(timeSeries[i + 1] - timeSeries[i], duration)   # phần "sở hữu" bị đòn sau cắt
return total + duration                                         # đòn cuối trọn vẹn
```

**Checklist trước khi submit**
- [ ] Đoạn độc là `[t, t + duration − 1]` — đúng `duration` giây, không lệch 1?
- [ ] Đã cộng `duration` cho **đòn cuối**?
- [ ] Vòng lặp dừng ở `len − 1` (không đọc vượt chỉ số)?
- [ ] Ca `duration = 0`, đòn trùng giây, một đòn duy nhất?
- [ ] Không mô phỏng từng giây khi `duration` tới `10^7`?